# Detector de balón rojo para el carro (FOMO, int8, ESP32)

Este cuaderno hace todo el camino del Machine Learning: **datos → entrenamiento → medición → exportar al microcontrolador**.

| Sección | Qué hace | Qué aprendes |
|---|---|---|
| 1. Datos | Carga tus fotos y etiquetas y las separa en entrenamiento / validación | Por qué se separa *por sesión* |
| 2. Etiquetas | Convierte cada círculo en un mapa de 12×16 celdas | Qué "respuesta" se le pide al modelo |
| 3. Aumentos | Crea variaciones de cada foto | Cómo evitar que el modelo memorice |
| 4. Modelo | MobileNetV2 recortado + una cabeza pequeña | Transfer learning, FOMO |
| 5. Medir | Acierto, falsa alarma, recall y el umbral | Cómo saber si el modelo sirve |
| 6. Entrenar | Ciclo de entrenamiento con curvas | Sobreajuste (overfitting) |
| 7. Resultado | Tabla de umbrales y comparación con el detector por COLOR | ¿Valió la pena el ML? |
| 8. Exportar | Cuantiza a int8 y escribe los archivos del firmware | Qué cambia al embeber |

**Cómo usarlo**

1. En tu Google Drive crea la carpeta `balon_rojo` y sube ahí `entrenamiento/comun.py` y la carpeta `datos/` del repositorio (con `fotos/` y `etiquetas.json`).
2. *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.
3. *Entorno de ejecución → Ejecutar todas*.

**¿Todavía no tienes fotos?** Pon `SOLO_SINTETICOS = True`: el cuaderno inventa imágenes de balones y corre de punta a punta (~10 min). Ese modelo **no** sirve para el carro, pero te deja ver y entender cada paso antes de juntar datos.

## 0. Configuración

In [ ]:
# ===================== Puedes cambiar estos valores =====================
CARPETA = '/content/drive/MyDrive/balon_rojo'   # en Drive: comun.py y datos/ (fotos/ + etiquetas.json)
SOLO_SINTETICOS = False   # True = sin fotos reales, solo imágenes inventadas (para aprender el proceso)
FRAC_SINTETICOS = 0.25    # fracción de cada lote con imágenes inventadas (complementan tus fotos)
ALPHA = 0.35              # ancho de MobileNetV2: 0.35 = rápido; 0.5 = más capacidad, más lento en la ESP32
SESIONES_VAL = []         # sesiones para medir, p. ej. ['cel03'] (la más parecida a la demo); [] = al azar
EPOCAS = 30               # vueltas de entrenamiento
PASOS = 100               # lotes por época
LOTE = 32                 # imágenes por lote
TASA = 1e-3               # tasa de aprendizaje inicial (qué tan grandes son los pasos al ajustar pesos)
FALSA_ALARMA_MAX = 0.05   # el umbral se elige para que en fotos SIN balón haya alarma en <= 5 %
SEMILLA = 0               # misma semilla = mismo resultado (sirve para comparar cambios)

## Entorno

In [ ]:
import os, sys, json, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
else:
    CARPETA = os.path.abspath('..')          # ejecutándolo en el PC desde entrenamiento/: usa el repositorio
os.makedirs(CARPETA, exist_ok=True)
sys.path[:0] = [CARPETA, os.path.join(CARPETA, 'entrenamiento')]
if EN_COLAB and not os.path.exists(os.path.join(CARPETA, 'comun.py')):
    print('No encontré comun.py en', CARPETA, '-> súbelo ahora (está en entrenamiento/comun.py)')
    for nombre, datos in files.upload().items():
        open(os.path.join(CARPETA, 'comun.py'), 'wb').write(datos)
import comun
from comun import ALTO, ANCHO, GH, GW, CELDA

DATOS = os.path.join(CARPETA, 'datos')
SALIDA = os.path.join(CARPETA, 'salida')
os.makedirs(SALIDA, exist_ok=True)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NINGUNA (Entorno de ejecución -> Cambiar tipo -> GPU)')

## 1. Datos

Cada foto se reduce a 160×120 (igual que hace la ESP32 al decodificar el JPEG a la mitad) y se guarda junto con sus círculos.

**Entrenamiento vs. validación.** El modelo aprende solo con las fotos de *entrenamiento*. Las de *validación* las usamos para medir. Si una foto de validación es casi idéntica a una de entrenamiento (fotos seguidas de la misma sesión), el examen sería "con las respuestas a la vista" y la medida saldría mejor de lo real. Por eso se separa **por sesión**: sesiones completas van a validación.

In [ ]:
lista = [] if SOLO_SINTETICOS else comun.cargar_etiquetas(DATOS)
if not lista and not SOLO_SINTETICOS:
    raise SystemExit(f'No hay fotos etiquetadas en {DATOS} (¿subiste datos/fotos y datos/etiquetas.json?). '
                     'Para probar sin fotos pon SOLO_SINTETICOS = True en la configuración.')
BASES, CIRCS, SESIONES = [], [], []
for ruta, circ in lista:
    bgr = cv2.imread(ruta)
    if bgr is None:
        print('No se pudo leer', ruta); continue
    b, c = comun.preparar_base(bgr, circ)
    BASES.append(b); CIRCS.append(c); SESIONES.append(comun.sesion_de(ruta))

rng = np.random.default_rng(SEMILLA)
sesiones = sorted(set(SESIONES))
if len(sesiones) >= 2:
    if SESIONES_VAL:
        faltan = set(SESIONES_VAL) - set(sesiones)
        assert not faltan, f'SESIONES_VAL tiene sesiones sin fotos etiquetadas: {sorted(faltan)}. Hay: {sesiones}'
        ses_val = set(SESIONES_VAL)
    else:
        rng.shuffle(sesiones)
        ses_val = set(sesiones[:max(1, round(len(sesiones) * 0.2))])
    es_val = np.array([s in ses_val for s in SESIONES])
    print('Sesiones de validación:', sorted(ses_val), '| de entrenamiento:', sorted(set(sesiones) - ses_val))
else:
    es_val = rng.random(len(BASES)) < 0.2
    if BASES:
        print('AVISO: hay una sola sesión, así que validación y entrenamiento tienen fotos casi iguales.\n'
              '       Las métricas saldrán optimistas. Toma otra sesión en otro lugar o con otra luz.')
IDX_TRAIN, IDX_VAL = np.where(~es_val)[0], np.where(es_val)[0]
for nombre, idx in (('Entrenamiento', IDX_TRAIN), ('Validación', IDX_VAL)):
    con = sum(len(CIRCS[i]) > 0 for i in idx)
    print(f'{nombre}: {len(idx)} fotos ({con} con balón, {len(idx) - con} sin balón)')

def ver_ejemplos(imgs, circs, titulo, n=12):
    fig, ax = plt.subplots(2, 6, figsize=(16, 4.5))
    for a, im, c in zip(ax.ravel(), imgs[:n], circs[:n]):
        a.imshow(im); a.axis('off')
        for x, y, r in c:
            a.add_patch(plt.Circle((x * im.shape[1], y * im.shape[0]), r * im.shape[1], fill=False, color='lime', lw=2))
    for a in ax.ravel()[len(imgs[:n]):]:
        a.axis('off')
    plt.suptitle(titulo); plt.show()

if BASES:
    m = rng.choice(len(BASES), min(12, len(BASES)), replace=False)
    ver_ejemplos([BASES[i] for i in m], [CIRCS[i] for i in m], 'Tus fotos con sus etiquetas (verde)')
r_demo = np.random.default_rng(5)
ej = [comun.balon_sintetico(r_demo) for _ in range(12)]
ver_ejemplos([e[0] for e in ej], [e[1] for e in ej], 'Imágenes SINTÉTICAS (inventadas): balones rojos, cosas rojas que no son balón y balones de otro color')

## 2. Etiquetas: el mapa de 12×16

El modelo no dibuja círculos: para cada celda de 8×8 píxeles dice la probabilidad de que **ahí haya balón** (esto se llama **FOMO**: *Faster Objects, More Objects*). Por eso la etiqueta de cada foto se convierte en un mapa de 12×16:

- **Verde (1):** celdas cuyo centro cae dentro del balón.
- **Rojo (se ignora, peso 0):** el borde del balón, que es ambiguo, y balones diminutos. Ni premia ni castiga.
- **Resto (0):** "aquí no hay balón".

In [ ]:
def ver_etiquetas(imgs, circs, titulo, mapas=None):
    fig, ax = plt.subplots(2, 6, figsize=(16, 4.5))
    for k, (a, im, c) in enumerate(zip(ax.ravel(), imgs, circs)):
        y = mapas[k] if mapas is not None else comun.etiquetas_mapa(c)
        capa = np.zeros((GH, GW, 4)); capa[..., 1] = y[..., 0]; capa[..., 0] = y[..., 1] == 0
        capa[..., 3] = 0.5 * np.maximum(y[..., 0], y[..., 1] == 0)
        a.imshow(im); a.imshow(cv2.resize(capa, (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST)); a.axis('off')
    for a in ax.ravel()[len(imgs):]:
        a.axis('off')
    plt.suptitle(titulo); plt.show()

if BASES:
    m = [i for i in range(len(BASES)) if len(CIRCS[i])][:12] or list(range(min(12, len(BASES))))
    ver_etiquetas([comun.base_a_entrada(BASES[i]) for i in m], [CIRCS[i] for i in m], 'Etiquetas de tus fotos (verde = balón, rojo = se ignora)')
ver_etiquetas([e[0] for e in ej], [e[1] for e in ej], 'Etiquetas de las sintéticas')

## 3. Aumentos de datos

Con unos cientos de fotos el modelo podría **memorizarlas** en vez de aprender "qué es un balón rojo". Para evitarlo, cada vez que una foto entra al entrenamiento se modifica al azar: recorte y zoom, espejo, brillo, contraste, saturación, desenfoque, ruido y compresión JPEG (todo eso pasa con la cámara del carro).

Ojo con lo que **no** se debe aumentar: si cambiáramos mucho el tono, el balón rojo se volvería naranja o rosado y le estaríamos enseñando que "cualquier color sirve". **Un aumento nunca debe cambiar la respuesta correcta.**

In [ ]:
def recorte_aleatorio(r, base, circ):
    """Zoom al azar: recorte de 70-100 % de la imagen 160x120 -> 128x96, con los círculos ajustados."""
    s = r.uniform(0.7, 1.0)
    cw, ch = int(comun.BASE_ANCHO * s), int(comun.BASE_ALTO * s)
    x0, y0 = int(r.integers(0, comun.BASE_ANCHO - cw + 1)), int(r.integers(0, comun.BASE_ALTO - ch + 1))
    im = cv2.resize(base[y0:y0 + ch, x0:x0 + cw], (ANCHO, ALTO), interpolation=cv2.INTER_AREA)
    c = circ.copy()
    if len(c):
        c[:, 0] = (c[:, 0] * comun.BASE_ANCHO - x0) / cw
        c[:, 1] = (c[:, 1] * comun.BASE_ALTO - y0) / ch
        c[:, 2] = c[:, 2] * comun.BASE_ANCHO / cw
    return im, c

def aumentar_color(r, im):
    x = im.astype(np.float32)
    x = x * r.uniform(0.9, 1.1, 3)                               # balance de blancos (poco: el rojo debe seguir rojo)
    gris = x.mean(-1, keepdims=True)
    x = gris + (x - gris) * r.uniform(0.7, 1.3)                  # saturación
    x = (x - 128) * r.uniform(0.7, 1.3) + 128 + r.uniform(-35, 35)   # contraste y brillo
    if r.random() < 0.3:
        x = cv2.GaussianBlur(x, (3, 3), 0)                       # desenfoque (carro en movimiento)
    if r.random() < 0.5:
        x += r.normal(0, r.uniform(2, 7), x.shape)               # ruido del sensor
    x = np.clip(x, 0, 255).astype(np.uint8)
    if r.random() < 0.5:                                         # compresión JPEG como la de la cámara
        ok, buf = cv2.imencode('.jpg', x[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, int(r.integers(25, 80))])
        x = cv2.imdecode(buf, cv2.IMREAD_COLOR)[..., ::-1]
    return np.ascontiguousarray(x)

def muestra(r, i):
    """i >= 0: foto real número i (aumentada). i < 0: imagen sintética nueva."""
    if i < 0:
        im, c = comun.balon_sintetico(r)
    else:
        im, c = recorte_aleatorio(r, BASES[i], CIRCS[i])
    if r.random() < 0.5:                                         # espejo horizontal
        im = im[:, ::-1]
        c = c.copy(); c[:, 0] = 1 - c[:, 0]
    return comun.a_entrada_modelo(aumentar_color(r, np.ascontiguousarray(im))), comun.etiquetas_mapa(c)

def lote(r, n=LOTE):
    if SOLO_SINTETICOS or len(IDX_TRAIN) == 0:
        idx = np.full(n, -1)
    else:
        n_s = int(n * FRAC_SINTETICOS)
        idx = np.concatenate([r.choice(IDX_TRAIN, n - n_s), np.full(n_s, -1)])
    xs, ys = zip(*(muestra(r, int(i)) for i in idx))
    return np.stack(xs), np.stack(ys)

x, y = lote(np.random.default_rng(7), 12)
ver_etiquetas([((xi + 1) * 127.5).astype(np.uint8) for xi in x], [None] * 12, 'Un lote de entrenamiento con aumentos y sus etiquetas', mapas=y)

## 4. El modelo

- **Transfer learning:** no partimos de cero. MobileNetV2 ya aprendió a ver bordes, texturas, formas y colores con 1.2 millones de fotos (ImageNet). Nos quedamos con sus primeras capas, hasta donde la imagen se reduce 8 veces: 96×128 → **12×16**, justo una celda por cada 8×8 píxeles.
- **Cabeza:** unas pocas capas nuevas que aprenden "¿esto es un balón rojo?" para cada celda. Terminan en una **sigmoide**, que da un número entre 0 y 1.
- **Sesgo inicial −3:** el modelo empieza diciendo "no hay balón" (p ≈ 5 %). Así no pierde tiempo al principio, porque casi todas las celdas son "no balón".
- **Pérdida focal:** le da más peso a los errores difíciles (una caja roja que cree que es balón) que a los fáciles (una pared blanca que ya reconoce).

In [ ]:
def construir_modelo(alpha=ALPHA):
    base = keras.applications.MobileNetV2(input_shape=(ALTO, ANCHO, 3), alpha=alpha,
                                          include_top=False, weights='imagenet')
    f8 = base.get_layer('block_6_expand_relu').output             # 12x16: la imagen reducida 8 veces
    x = layers.DepthwiseConv2D(3, padding='same', use_bias=False, name='cabeza_dw')(f8)
    x = layers.BatchNormalization(name='cabeza_bn')(x)
    x = layers.ReLU(6.0, name='cabeza_relu')(x)
    x = layers.Conv2D(32, 1, use_bias=False, name='cabeza_1x1')(x)
    x = layers.BatchNormalization(name='cabeza_1x1_bn')(x)
    x = layers.ReLU(6.0, name='cabeza_1x1_relu')(x)
    x = layers.Conv2D(1, 1, activation='sigmoid', name='mapa_balon',
                      bias_initializer=keras.initializers.Constant(-3.0))(x)
    return keras.Model(base.input, x, name='detector_balon')

def perdida_focal(peso_positivo=2.0, gamma=2.0):
    import keras.ops as K
    def perdida(y_true, y_pred):
        y, w = y_true[..., 0:1], y_true[..., 1:2]                  # etiqueta y peso de cada celda
        p = K.clip(y_pred, 1e-6, 1 - 1e-6)
        pt = y * p + (1 - y) * (1 - p)                              # probabilidad que el modelo le dio a la respuesta correcta
        a = y * peso_positivo + (1 - y)
        return K.sum(w * a * (1 - pt) ** gamma * -K.log(pt)) / (K.sum(w) + 1e-6)
    return perdida

modelo = construir_modelo()
print(f'Parámetros: {modelo.count_params():,} | entrada {modelo.input.shape} -> salida {modelo.output.shape}')
print(f'En int8 cada peso ocupa 1 byte: ~{modelo.count_params() // 1024} KB de pesos. El archivo final pesa más '
      '(unos 60-70 KB) porque guarda también la estructura y las escalas de cada capa.')

## 5. Cómo medir si sirve

Lo que importa en el carro no es la pérdida sino:

- **Acierto:** en fotos con balón, ¿la mancha principal cae sobre un balón? Es la que el carro va a seguir.
- **Falsa alarma:** en fotos **sin** balón, ¿aparece alguna mancha? Eso haría que el carro persiga algo que no es el balón.
- **Recall:** de todos los balones, ¿cuántos ve?

El **umbral** (desde qué probabilidad se "enciende" una celda) es un trato entre ambos: si lo subes hay menos falsas alarmas pero ve menos balones. Aquí no se elige a ojo: se calcula el menor umbral que deja la falsa alarma en `FALSA_ALARMA_MAX` en las fotos sin balón de validación.

Abajo también se mide el **detector por COLOR** (sin ML) sobre las mismas fotos: es la línea base que el modelo debe superar.

In [ ]:
imgs, CIRCS_V, grupo = [], [], []
for i in IDX_VAL:
    imgs.append(comun.base_a_entrada(BASES[i])); CIRCS_V.append(CIRCS[i]); grupo.append('real')
r_val = np.random.default_rng(123)
for k in range(300 if (SOLO_SINTETICOS or FRAC_SINTETICOS > 0) else 0):
    im, c = comun.balon_sintetico(r_val); imgs.append(im); CIRCS_V.append(c); grupo.append('sintetica')
IMG_V = np.stack(imgs)
X_V = comun.a_entrada_modelo(IMG_V)
Y_V = np.stack([comun.etiquetas_mapa(c) for c in CIRCS_V])
GRUPO_V = np.array(grupo)
PRINCIPAL = 'real' if (GRUPO_V == 'real').any() else 'sintetica'   # las métricas que mandan
CON_BALON = np.array([any(r * ANCHO >= 2 and 0 <= x < 1 and 0 <= y < 1 for x, y, r in c) for c in CIRCS_V])
print(f'Validación: {len(X_V)} imágenes | se decide con las "{PRINCIPAL}": '
      f'{((GRUPO_V == PRINCIPAL) & CON_BALON).sum()} con balón, {((GRUPO_V == PRINCIPAL) & ~CON_BALON).sum()} sin balón')

def cae_en_balon(b, circ):
    return any(np.hypot((b['cx'] - x) * ANCHO, (b['cy'] - y) * ALTO) <= r * ANCHO + CELDA for x, y, r in circ)

def predecir(m, X):
    return m.predict(X, batch_size=128, verbose=0)[..., 0]

def medir(mapas, umbral, grupo=None):
    sel = GRUPO_V == (grupo or PRINCIPAL)
    blobs = [comun.decodificar(mp, umbral) for mp in mapas]
    pos, neg = np.where(sel & CON_BALON)[0], np.where(sel & ~CON_BALON)[0]
    vistos = perdidos = falsos = 0
    for i in np.where(sel)[0]:
        for x, y, r in CIRCS_V[i]:
            if r * ANCHO < 2 or not (0 <= x < 1 and 0 <= y < 1):
                continue
            if any(np.hypot((b['cx'] - x) * ANCHO, (b['cy'] - y) * ALTO) <= r * ANCHO + CELDA for b in blobs[i]):
                vistos += 1
            else:
                perdidos += 1
        falsos += sum(not cae_en_balon(b, CIRCS_V[i]) for b in blobs[i])
    return dict(acierto=float(np.mean([bool(blobs[i]) and cae_en_balon(blobs[i][0], CIRCS_V[i]) for i in pos])) if len(pos) else 0.0,
                falsa_alarma=float(np.mean([len(blobs[i]) > 0 for i in neg])) if len(neg) else 0.0,
                recall=vistos / max(vistos + perdidos, 1),
                falsos_por_foto=falsos / max(sel.sum(), 1))

def umbral_para(mapas, fa=FALSA_ALARMA_MAX):
    neg = (GRUPO_V == PRINCIPAL) & ~CON_BALON
    if neg.sum() < 10:
        return 0.5      # muy pocas fotos sin balón para calcularlo: toma más fotos SIN balón
    return float(np.clip(np.quantile(mapas[neg].max(axis=(1, 2)), 1 - fa), 0.2, 0.95))

def resumen(m, u):
    return (f"umbral {u:.2f} | acierto {m['acierto']:.1%} | falsa alarma {m['falsa_alarma']:.1%} | "
            f"recall {m['recall']:.1%} | manchas falsas por foto {m['falsos_por_foto']:.2f}")

MAPAS_COLOR = np.stack([comun.mapa_rojo(im, 60) for im in IMG_V])
print('Línea base, detector por COLOR (delta 60, umbral 0.30):', resumen(medir(MAPAS_COLOR, 0.30), 0.30))

## 6. Entrenar

En cada época:

1. Se entrena con `PASOS` lotes. En cada lote el modelo predice, se calcula la pérdida y se ajustan los pesos (**descenso de gradiente**).
2. Se mide en validación: la pérdida, el umbral calculado y las métricas.
3. Si es el mejor hasta ahora (acierto − falsa alarma), se guarda. Si empata con el mejor, se guarda solo si su pérdida de validación es menor.

La tasa de aprendizaje baja poco a poco (*cosine decay*): pasos grandes al principio y pequeños al final para afinar. Con GPU T4 cada época tarda unos 10 a 30 segundos.

In [ ]:
modelo.compile(optimizer=keras.optimizers.Adam(keras.optimizers.schedules.CosineDecay(TASA, EPOCAS * PASOS, alpha=0.05)),
               loss=perdida_focal())
RUTA_MEJOR = os.path.join(SALIDA, 'modelo_balon_mejor.keras')
HIST, mejor, mejor_val = [], -1.0, float('inf')
r_train = np.random.default_rng(SEMILLA)
for ep in range(EPOCAS):
    t0 = time.time()
    perd = [float(modelo.train_on_batch(*lote(r_train))) for _ in range(PASOS)]
    perd_val = float(modelo.evaluate(X_V, Y_V, batch_size=128, verbose=0))
    mapas = predecir(modelo, X_V)
    u = umbral_para(mapas)
    m = medir(mapas, u)
    puntaje = m['acierto'] - m['falsa_alarma']
    marca = ''
    # Con pocas fotos de validación el puntaje se satura (p. ej. 1.0); en un empate gana la menor pérdida
    if puntaje > mejor or (puntaje == mejor and perd_val < mejor_val):
        mejor, mejor_val = puntaje, perd_val; modelo.save(RUTA_MEJOR); marca = '  <- guardado'
    HIST.append(dict(epoca=ep + 1, perdida=np.mean(perd), perdida_val=perd_val, umbral=u, **m))
    print(f'época {ep + 1:2d}/{EPOCAS} pérdida {np.mean(perd):.4f} (val {perd_val:.4f}) | {resumen(m, u)} '
          f'({time.time() - t0:.0f} s){marca}', flush=True)

In [ ]:
# Curvas de aprendizaje
e = [h['epoca'] for h in HIST]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(e, [h['perdida'] for h in HIST], label='entrenamiento'); ax[0].plot(e, [h['perdida_val'] for h in HIST], label='validación')
ax[0].set_title('Pérdida (más bajo = mejor)'); ax[0].set_xlabel('época'); ax[0].legend()
for k in ('acierto', 'falsa_alarma', 'recall'):
    ax[1].plot(e, [h[k] for h in HIST], label=k)
ax[1].set_title('Métricas en validación'); ax[1].set_xlabel('época'); ax[1].legend(); plt.show()

**Cómo leer las curvas**

- Las dos pérdidas bajan juntas: el modelo está aprendiendo cosas que sirven fuera de las fotos de entrenamiento. ✅
- La de entrenamiento sigue bajando y la de validación **sube**: el modelo está **memorizando** (sobreajuste u *overfitting*). Soluciones: más fotos y más variadas, más aumentos, menos épocas.
- Las dos se quedan altas: el modelo no logra aprender (*underfitting*). Revisa las etiquetas, sube `ALPHA` o entrena más.

## 7. Resultado final: umbral y comparación con el detector por COLOR

In [ ]:
modelo = keras.models.load_model(RUTA_MEJOR, compile=False)
MAPAS_F = predecir(modelo, X_V)
UMBRAL = umbral_para(MAPAS_F)
print(f'Umbral elegido: {UMBRAL:.2f} (falsa alarma <= {FALSA_ALARMA_MAX:.0%} en fotos sin balón)\n')
print(f"{'umbral':>7} | {'acierto':>8} | {'falsa alarma':>12} | {'recall':>7}")
for uu in sorted({0.3, 0.4, 0.5, 0.6, 0.7, 0.8, round(UMBRAL, 2)}):
    mm = medir(MAPAS_F, uu)
    print(f"{uu:>7.2f} | {mm['acierto']:>8.1%} | {mm['falsa_alarma']:>12.1%} | {mm['recall']:>7.1%}" + ('  <- elegido' if uu == round(UMBRAL, 2) else ''))
print('\nMODELO:', resumen(medir(MAPAS_F, UMBRAL), UMBRAL))
print('COLOR: ', resumen(medir(MAPAS_COLOR, 0.30), 0.30))
if PRINCIPAL == 'real' and (GRUPO_V == 'sintetica').any():
    print('(en las sintéticas, el modelo:', resumen(medir(MAPAS_F, UMBRAL, 'sintetica'), UMBRAL) + ')')

def ver_detecciones(indices, titulo):
    fig, ax = plt.subplots(2, 6, figsize=(16, 4.8))
    for a, i in zip(ax.ravel(), indices):
        a.imshow(IMG_V[i]); a.imshow(cv2.resize(MAPAS_F[i], (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST), cmap='jet', alpha=0.4, vmin=0, vmax=1)
        for x, y, r in CIRCS_V[i]:
            a.add_patch(plt.Circle((x * ANCHO, y * ALTO), r * ANCHO, fill=False, color='lime', lw=1.5))
        for k, b in enumerate(comun.decodificar(MAPAS_F[i], UMBRAL)):
            a.plot(b['cx'] * ANCHO, b['cy'] * ALTO, 'x', color='white' if k == 0 else 'yellow', ms=10, mew=3)
        a.axis('off')
    for a in ax.ravel()[len(indices):]:
        a.axis('off')
    plt.suptitle(titulo); plt.show()

sel = np.where(GRUPO_V == PRINCIPAL)[0]
errores = [i for i in sel if (CON_BALON[i] and not (comun.decodificar(MAPAS_F[i], UMBRAL) and cae_en_balon(comun.decodificar(MAPAS_F[i], UMBRAL)[0], CIRCS_V[i])))
           or (not CON_BALON[i] and comun.decodificar(MAPAS_F[i], UMBRAL))]
ver_detecciones(list(sel[:12]), 'Validación: verde = etiqueta, X blanca = mancha principal, color = mapa del modelo')
if errores:
    ver_detecciones(errores[:12], f'ERRORES ({len(errores)}): mira qué tienen en común -> esas son las fotos que te faltan')

## 8. Exportar a int8 para la ESP32

1. Las capas `ZeroPadding2D` se reemplazan por `padding='same'` (mismo resultado, menos operaciones).
2. **Cuantización int8:** los pesos y activaciones pasan de números decimales de 32 bits a enteros de 8 bits. El modelo queda 4 veces más pequeño y la ESP32 lo corre mucho más rápido. Para elegir las escalas, el convertidor mira fotos reales de **calibración**.
3. Se comprueba que el modelo use solo operaciones que el firmware registró en `detector.cc`.
4. Se mide de nuevo en int8, porque la cuantización puede empeorar un poco el resultado.
5. Se escriben `modelo_balon.cc/.h` (el modelo como arreglo de bytes en C, más el umbral) y `pruebas_vectores.h` (para la autoprueba de la ESP32).

In [ ]:
def sin_padding(m):
    def clonar(capa):
        if isinstance(capa, layers.ZeroPadding2D):
            return layers.Identity(name=capa.name)
        cfg = capa.get_config()
        if isinstance(capa, (layers.DepthwiseConv2D, layers.Conv2D)) and cfg.get('padding') == 'valid' \
                and tuple(cfg.get('strides', (1, 1))) == (2, 2):
            cfg['padding'] = 'same'
        return capa.__class__.from_config(cfg)
    nuevo = keras.models.clone_model(m, clone_function=clonar)
    for c in m.layers:
        if c.get_weights():
            nuevo.get_layer(c.name).set_weights(c.get_weights())
    return nuevo

m2 = sin_padding(modelo)
d = np.abs(modelo.predict(X_V[:64], verbose=0) - m2.predict(X_V[:64], verbose=0)).max()
print(f'Diferencia original vs sin padding: {d:.2e}'); assert d < 1e-4

cal = list(X_V[np.random.default_rng(1).choice(len(X_V), min(200, len(X_V)), replace=False)])
r_cal = np.random.default_rng(2)
while len(cal) < 300:
    cal += list(lote(r_cal)[0])
def representativo():
    for x in cal:
        yield [x[None]]
conv = tf.lite.TFLiteConverter.from_keras_model(m2)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representativo
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
TFL = conv.convert()
print(f'Modelo int8: {len(TFL):,} bytes')

it = tf.lite.Interpreter(model_content=TFL, experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_REF)
it.allocate_tensors()
ops = sorted({o['op_name'] for o in it._get_ops_details()})
SOPORTADAS = {'CONV_2D', 'DEPTHWISE_CONV_2D', 'ADD', 'LOGISTIC', 'PAD', 'QUANTIZE', 'DEQUANTIZE', 'RESHAPE',
              'RELU6', 'MAX_POOL_2D', 'CONCATENATION', 'RESIZE_NEAREST_NEIGHBOR'}   # = las de detector.cc
print('Operaciones del modelo:', ', '.join(ops))
assert set(ops) <= SOPORTADAS, f'El firmware no tiene: {set(ops) - SOPORTADAS} -> agrégalas en detector.cc'
ent, sal = it.get_input_details()[0], it.get_output_details()[0]
s_in, z_in = ent['quantization']; s_out, z_out = sal['quantization']
print(f"Entrada {ent['shape']} escala {s_in:.6f} cero {z_in} | Salida {sal['shape']} escala {s_out:.6f} cero {z_out}")
LUT = np.clip(np.round((np.arange(256) / 127.5 - 1.0) / s_in + z_in), -128, 127).astype(np.int8)   # = s_lut en detector.cc

def correr_int8(img_uint8):
    it.set_tensor(ent['index'], LUT[img_uint8][None])
    it.invoke()
    return it.get_tensor(sal['index'])[0]

MAPAS_Q = np.stack([np.clip((correr_int8(im)[..., 0].astype(np.float32) - z_out) * s_out, 0, 1) for im in IMG_V])
print('float:', resumen(medir(MAPAS_F, UMBRAL), UMBRAL))
print('int8: ', resumen(medir(MAPAS_Q, UMBRAL), UMBRAL))
print(f'Diferencia media float vs int8 por celda: {np.abs(MAPAS_F - MAPAS_Q).mean():.4f}')

# --- Archivos para el firmware
open(os.path.join(SALIDA, 'modelo_balon_int8.tflite'), 'wb').write(TFL)
with open(os.path.join(SALIDA, 'modelo_balon.h'), 'w') as f:
    f.write('// Generado por el cuaderno detector_balon.ipynb — no editar a mano\n#pragma once\n#include <stdint.h>\n'
            f'#define MODELO_UMBRAL {UMBRAL:.3f}f   // falsa alarma <= {FALSA_ALARMA_MAX:.0%} en las fotos sin balón de validación\n'
            'extern const uint8_t g_modelo_balon[];\nextern const unsigned int g_modelo_balon_len;\n')
with open(os.path.join(SALIDA, 'modelo_balon.cc'), 'w') as f:
    f.write('// Generado por el cuaderno detector_balon.ipynb — no editar a mano\n'
            f'// Detector de balón FOMO, MobileNetV2 {ALPHA} recortado, int8, entrada 96x128x3\n'
            '#include "modelo_balon.h"\n\nalignas(16) const uint8_t g_modelo_balon[] = {\n')
    for i in range(0, len(TFL), 16):
        f.write('  ' + ', '.join(f'0x{b:02x}' for b in TFL[i:i + 16]) + ',\n')
    f.write(f'}};\nconst unsigned int g_modelo_balon_len = {len(TFL)};\n')
vec = ([i for i in sel if CON_BALON[i]][:2] + [i for i in sel if not CON_BALON[i]][:1]) or list(range(min(3, len(IMG_V))))
with open(os.path.join(SALIDA, 'pruebas_vectores.h'), 'w') as f:
    f.write('// Generado por el cuaderno detector_balon.ipynb: imágenes de validación (RGB) y la salida int8 esperada\n'
            f'#pragma once\n#include <stdint.h>\n#define N_VECTORES {len(vec)}\n'
            'static const uint8_t VECTOR_ENTRADA[N_VECTORES][96*128*3] = {\n')
    for i in vec:
        f.write('{' + ','.join(map(str, IMG_V[i].ravel())) + '},\n')
    f.write('};\nstatic const int8_t VECTOR_SALIDA[N_VECTORES][12*16] = {\n')
    for i in vec:
        f.write('{' + ','.join(map(str, correr_int8(IMG_V[i]).ravel())) + '},\n')
    f.write('};\n')
print('Escritos en', SALIDA, ':', sorted(os.listdir(SALIDA)))
if EN_COLAB:
    for n in ['modelo_balon.cc', 'modelo_balon.h', 'pruebas_vectores.h', 'modelo_balon_int8.tflite']:
        files.download(os.path.join(SALIDA, n))

**Dónde va cada archivo**

| Archivo | Carpeta del repositorio |
|---|---|
| `modelo_balon.cc`, `modelo_balon.h`, `pruebas_vectores.h` | `firmware_esp32/main/` (reemplazan a los marcadores) |
| `modelo_balon_int8.tflite` | `entrenamiento/` (para `probar_pc.py --modo modelo`) |

Luego ejecuta `idf.py build flash monitor`. Al arrancar, el monitor debe decir `AUTOPRUEBA DEL MODELO: OK`, y en la página del carro se habilita el botón **Modo MODELO**.

## 9. Probar con fotos tuyas

Sube fotos (mejor si son de la cámara del carro, con `capturar_fotos.py`). Se muestra el mapa del modelo **int8**, el mismo que calcula la ESP32.

In [ ]:
def mostrar(rgb, titulo=''):
    mapa = np.clip((correr_int8(np.ascontiguousarray(rgb))[..., 0].astype(np.float32) - z_out) * s_out, 0, 1)
    blobs = comun.decodificar(mapa, UMBRAL)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].imshow(rgb); ax[0].set_title(titulo); ax[0].axis('off')
    ax[1].imshow(rgb); ax[1].imshow(cv2.resize(mapa, (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST), cmap='jet', alpha=0.45, vmin=0, vmax=1)
    for k, b in enumerate(blobs):
        ax[1].add_patch(plt.Circle((b['cx'] * ANCHO, b['cy'] * ALTO), 9 if k == 0 else 5, fill=False, color='lime' if k == 0 else 'yellow', lw=2))
    ax[1].set_title('balones: ' + (', '.join(f"x={b['cx']:.2f} área={b['area']}" for b in blobs) or 'ninguno'), fontsize=9)
    ax[1].axis('off'); plt.show()

if EN_COLAB:
    for nombre, datos in files.upload().items():
        bgr = cv2.imdecode(np.frombuffer(datos, np.uint8), cv2.IMREAD_COLOR)
        if bgr is not None:
            mostrar(comun.preparar(bgr), nombre)

## Para pensar (y experimentar)

1. Entrena con `SOLO_SINTETICOS = True` y prueba ese modelo con fotos reales. ¿Qué pasa y por qué?
2. Pon `FRAC_SINTETICOS = 0` y luego `0.5`. ¿Las sintéticas ayudan o estorban con tus fotos?
3. Cambia `FALSA_ALARMA_MAX` a 0.01 y a 0.2. ¿Cómo cambian el umbral y el acierto? ¿Qué prefieres en el carro?
4. Prueba `ALPHA = 0.5`. ¿Mejora el acierto? Mira en el monitor de la ESP32 cuánto más tarda (`ms`).
5. Mira la figura de ERRORES: ¿qué tienen en común? Toma más fotos de esas situaciones, etiquétalas y vuelve a entrenar. **Ese ciclo es el trabajo real del ML.**